### Understanding the propagation of half filled and full filled states through 1D XYZ Hamiltonian

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{j} (J_{x} \sigma_j^{x} \sigma_{j+1}^{x} + J_{y} \sigma_j^{y} \sigma_{j+1}^{y} + J_{z} \sigma_j^{z} \sigma_{j+1}^{z}) + \sum_{j} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z})}
$$

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import SuzukiTrotter, LieTrotter

from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
    generate_xyz_hamiltonian
)

import numpy as np
import scipy


#### I. Construct the XYZ Hamiltonian simulation circuit

In [ ]:
num_qubits = 80 # increase the number to the largest linear chain you can get in the IBM Quantum Computer
Jx, Jy, Jz = np.pi/8, np.pi/4, np.pi/2 # feel free to change these parameters and see how the results change
h_x, h_y, h_z = np.pi/3, np.pi/6, np.pi/9 # feel free to change these parameters and see how the results change
dt = 0.01 # time evolution of each trotter step; feel free to increase or decrease and see how the results change
num_trotter_steps = 10 # decide on the number of trotter steps; do some trial and error to find the 2-qubit depth of the circuit for different trotter steps

Build a 50-site linear coupling map and construct the Hamiltonian from it

In [ ]:
coupling_map = CouplingMap.from_line(num_qubits)

In [ ]:
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(Jx, Jy, Jz),
    ext_magnetic_field=(h_x, h_y, h_z),
)
print(hamiltonian)

Construct a half-filled initial state (also called Neel State). Note a Neel state looks like 010101...

In [ ]:
init_state_neel = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    if i%2 == 0:
        init_state.x(i)

Construct a full-filled initial state. Note a full filled state looks like 111111...

In [ ]:
init_state_full = 

Create Hamiltonian simulation circuit with Lie Trotter decomposition

In [ ]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=dt*num_trotter_steps, # total time of evolution
    synthesis=LieTrotter(reps=num_trotter_steps), 
    # you should also redo the excersize with SuzukiTrotter; compare the depth and results of the two synthesis methods
)

Compose this circuit with the initial state

In [ ]:
circuit = init_state_neel.compose(circuit)

#### II. Transpile the circuit to the backend

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
service = QiskitRuntimeService()
backend = service.least_busy(min_num_qubits=127)
backend

Transpile the circuit and print the 2-qubit depth and the number of 2-qubit gates. Remember, a utility-scale work resides at the regime of ~100 qubits and ~5000 2-qubit gates.

In [ ]:
from qiskit import generate_preset_pass_manager
pm = 
isa_circuit = pm.run(circuit)

#### III. Execute on IBM backend

In [ ]:
from qiskit_ibm_runtime import EstimatorV2
estimator = EstimatorV2(mode=backend)

We want to calculate the probability that each site is occupied (i.e., probability of the qubit being in state `1`). We can do that by using Sampler. But with Sampler we cannot apply different error mitigation techniques.

**Food for thought**: What observable should I use so that I can extract the probability of the qubit being in state `1`?

In [ ]:
from qiskit.quantum_info import SparsePauliOp
observable = []

Apply the layout of the transpiled circuit to the observable

In [ ]:
isa_observable = [obs.apply_layout(isa_circuit.layout) for obs in observable]


In [ ]:
pub = (isa_circuit, isa_observable)

In [ ]:
job = estimator.run([pub])
result = job.result()[0]

# obtain the expvals for each observable

#### IV. Postprocess

In [ ]:
### Obtain the probability of `1` for each qubit
prob_occupied = []

### Using error mitigation

The results from the hardware are noisy, and therefore may not be perfectly reliable. But we can use error mitigation to account for the noise. We shall use Probabilistic Error Amplification (PEA) and Twirled Readout Error Extinction (TREX) for this experiment. The first one accounts for Gate Errors, while the second one accounts for SPAM error.

For PEA, it is necessary to first learn the noise in the system. This can be done via NoiseLearner: https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/noise-learner-noise-learner. Also look into this tutorial: https://quantum.cloud.ibm.com/docs/en/tutorials/probabilistic-error-amplification#learn-the-noise-model-for-pea

In [ ]:
### Learn the noise in the system by running NoiseLearner

In [ ]:
### Run the circuit with PEA and TREX error mitigations and calculate the probability of occupancy for each qubit

### Using good noise factors for PEA

A general issue with PEA is that if you use any random noise factor, the result may not improve. Therefore, it is important to understand which noise factors and extrapolators are good for the given circuit. A method to do that is to cliffordize the circuit, calculate the ideal result for the clifford circuit, try out different noise factors on the clifford circuit and select the one with the best result.

This can be performed using the NEAT tool: https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/debug-tools-neat

In [ ]:
#### Cliffordize the Hamiltonian simulation circuit

In [ ]:
### Calculate the probability of occupancy for the clifford circuit

In [ ]:
### Test for different noise factors and extrapolators
### Test with (1,3,5), (1,2,3), (1,1.2,1.4), (1,1.1,1.2) and for extrapolators linear, quadratic and exponential
### Calculate the mitigated probabilities of occupancy for each qubit and compare with the ideal values

In [ ]:
### Use the best noise factor and extrapolator to calculate the mitigated probabilities of occupancy for the original Hamiltonian simulation circuit

### Comparing against ideal solution

At 80 qubits, the ideal expectation values are not known. One method to simulate quantum circuits is Pauli Propagation: https://github.com/Qiskit/pauli-prop
In this part, you will implement the same problem in Pauli Propagation; and compare the result as well as execution time with QPU.